# H100 multi-split and extended Peptides reproduction

This operator notebook runs 180 heterophily raw jobs (official splits 0/1/2, seed 25) and 10 task-correct Peptides jobs, then verifies 70 aggregate summaries. Raw runs resume independently and conflicting completed artifacts fail unless `RERUN=True`. The four frozen legacy result directories are never written.

In [ ]:
# GPU isolation must happen before importing torch or the project modules.
import os, subprocess
listing = subprocess.run(['nvidia-smi', '-L'], check=True, capture_output=True, text=True).stdout
gpu_lines = [line for line in listing.splitlines() if line.strip().startswith('GPU ')]
if not gpu_lines:
    raise RuntimeError('nvidia-smi found no GPUs')
SELECTED_GPU = int(os.environ.get('GBDN_H100_INDEX', len(gpu_lines) - 1))
if SELECTED_GPU < 0 or SELECTED_GPU >= len(gpu_lines):
    raise ValueError(f'invalid GBDN_H100_INDEX={SELECTED_GPU}')
if 'H100' not in gpu_lines[SELECTED_GPU].upper():
    raise RuntimeError(f'selected GPU is not H100: {gpu_lines[SELECTED_GPU]}')
os.environ['CUDA_VISIBLE_DEVICES'] = str(SELECTED_GPU)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print(f'Physical H100 selected; subprocesses will see only cuda:0: {gpu_lines[SELECTED_GPU]}')

In [ ]:
from pathlib import Path
import html, json, subprocess, sys
import torch
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'reproduce_legacy.py').is_file():
    ROOT = ROOT.parent
RUNNER = ROOT / 'scripts' / 'reproduce_legacy.py'
DATA_ROOT = ROOT / 'data' / 'legacy'
HETERO_OUTPUT = ROOT / 'results_multisplit'
PEPTIDE_OUTPUT = ROOT / 'results_LRGB_extended'
HETERO_DATASETS = ['Roman-empire', 'Amazon-ratings', 'Minesweeper', 'Tolokers', 'Questions']
HETERO_MODELS = ['GBDN+', 'ChebNet', 'ChebNetII', 'H2GCN', 'FAGCN', 'MLP', 'MixHop', 'GAT', 'GraphSAGE', 'ADGN', 'ResNet', 'ResNet+SGC']
PEPTIDE_DATASETS = ['Peptides-func', 'Peptides-struct']
PEPTIDE_MODELS = ['GCN', 'GINE', 'GAT', 'ChebNet_K10', 'GBDN+']
SPLITS = [0, 1, 2]
SEED = 25
WORKERS = 'auto'
SMOKE_MODE = True
RUN_FULL = True
RERUN = False

assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert 'H100' in torch.cuda.get_device_name(0).upper()
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

In [ ]:
def stream_command(arguments):
    command = [sys.executable, str(RUNNER), *map(str, arguments)]
    print(' '.join(command))
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

common = ['--data-root', DATA_ROOT, '--output-root', HETERO_OUTPUT, '--lrgb-output-root', PEPTIDE_OUTPUT, '--seed', SEED]
if RERUN:
    common.append('--rerun')
if SMOKE_MODE:
    stream_command(['smoke', *common])

In [ ]:
if RUN_FULL:
    stream_command([
        'run-all', *common, '--splits', *SPLITS, '--workers', WORKERS,
        '--heterophily-datasets', *HETERO_DATASETS, '--heterophily-models', *HETERO_MODELS,
        '--peptides-datasets', *PEPTIDE_DATASETS, '--peptides-models', *PEPTIDE_MODELS,
    ])
else:
    print('RUN_FULL=False: only the smoke check was requested.')

In [ ]:
def display_records(records):
    headers = list(records[0])
    rows = ''.join('<tr>' + ''.join(f'<td>{html.escape(str(row[key]))}</td>' for key in headers) + '</tr>' for row in records)
    display(HTML('<table><thead><tr>' + ''.join(f'<th>{html.escape(key)}</th>' for key in headers) + '</tr></thead><tbody>' + rows + '</tbody></table>'))

REPORT = ROOT / 'h100_multisplit_report.md'
stream_command(['report', '--output-root', HETERO_OUTPUT, '--lrgb-output-root', PEPTIDE_OUTPUT, '--output', REPORT])

hetero_rows = []
for dataset in HETERO_DATASETS:
    for model in HETERO_MODELS:
        item = json.loads((HETERO_OUTPUT / dataset / f'{model}.json').read_text())
        hetero_rows.append({'dataset': dataset, 'model': model, 'test_auroc_mean': item['metrics']['test_auroc']['mean'], 'test_auroc_std': item['metrics']['test_auroc']['std'], 'test_acc_mean': item['metrics']['test_acc']['mean'], 'runs': item['run_count']})
display_records(hetero_rows)

peptide_rows = []
for dataset in PEPTIDE_DATASETS:
    metric = 'test_ap' if dataset == 'Peptides-func' else 'test_mae'
    for model in PEPTIDE_MODELS:
        item = json.loads((PEPTIDE_OUTPUT / dataset / f'{model}.json').read_text())
        peptide_rows.append({'dataset': dataset, 'model': model, 'metric': metric, 'value': item['metrics'][metric]['mean'], 'runs': item['run_count']})
display_records(peptide_rows)

In [ ]:
# Final acceptance gate: recomputes all raw metrics and summary statistics.
stream_command(['verify', '--output-root', HETERO_OUTPUT, '--lrgb-output-root', PEPTIDE_OUTPUT, '--splits', *SPLITS, '--seed', SEED])
print('PASS: 190 raw runs and 70 aggregate summaries are complete and internally consistent.')